In [2]:
import pandas as pd
import numpy as np
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint as sp_randint
from scipy.stats import uniform
from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer, SimpleImputer
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from scipy.stats import randint

#RandomForestRegressor
from sklearn.ensemble import RandomForestClassifier
import pandas as pd
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.ensemble import RandomForestRegressor
# Ensure other necessary imports are also included
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score



In [2]:
train=pd.read_csv('train.csv')
name=train['Name']
#extract title from name
train['Title']=name.str.extract(r'([A-Za-z]+)\.', expand=False)
train['Title'].value_counts()


Title
Mr          517
Miss        182
Mrs         125
Master       40
Dr            7
Rev           6
Mlle          2
Major         2
Col           2
Countess      1
Capt          1
Ms            1
Sir           1
Lady          1
Mme           1
Don           1
Jonkheer      1
Name: count, dtype: int64

In [3]:
#relationship between title and survival
titleinfo=train[['Title', 'Survived']].groupby(['Title'], as_index=False).mean().sort_values(by='Survived', ascending=False)
titleinfo


,Title,Survived
16,Sir,1.000000
2,Countess,1.000000
14,Ms,1.000000
11,Mme,1.000000
6,Lady,1.000000
10,Mlle,1.000000
13,Mrs,0.792000
9,Miss,0.697802
8,Master,0.575000
1,Col,0.500000


In [41]:
#Preprocessing
#import data
train=pd.read_csv('train.csv')
test=pd.read_csv('test.csv')
    
 
# clean data
def clean(df):
    # Extract only prefix from Name and just keep prefix
    df['Name'] = df['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False)
    # Drop ticket
    df.drop('Ticket', axis=1, inplace=True)
    # Split cabin into letter and number
    df['CabinClass'] = df['Cabin'].str[0]
    #drop
    df.drop('Cabin', axis=1, inplace=True)
    med_fare = df.groupby(['Pclass', 'Parch', 'SibSp']).Fare.median()[3][0][0]
    #Filling the missing value in Fare with the median Fare of 3rd class alone passenger
    df['Fare'] = df['Fare'].fillna(med_fare)
    #Drop Fare
    #put them in 13 quantile based bin and put labels as 0-12


    df['FareBin']=pd.qcut(df['Fare'], 13, labels=False)

    #Age
    df['Age'] = df.groupby(['Sex', 'Pclass'])['Age'].transform(lambda x: x.fillna(x.median()))
    #put them in 10 quantile based bin
    df['AgeBin'] = pd.qcut(df['Age'], 10, labels=False)
    #drop Age
    df.drop('Age', axis=1, inplace=True)
    #Cabin Class
    df['Deck'] = df['CabinClass']
    #drop
    df.drop('CabinClass', axis=1, inplace=True)
    idx = df[df['Deck'] == 'T'].index
    df.loc[idx, 'Deck'] = 'A'

    df['Deck'] = df['Deck'].replace(['A', 'B', 'C'], 'ABC')
    df['Deck'] = df['Deck'].replace(['D', 'E'], 'DE')
    df['Deck'] = df['Deck'].replace(['F', 'G'], 'FG')
    df['Deck'] = df['Deck'].fillna('M')

    # Filling the missing values in Embarked with S
    df['Embarked'] = df['Embarked'].fillna('S')

    # Family size
    df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
    # Drop SibSp and Parch
    df.drop(['SibSp', 'Parch'], axis=1, inplace=True)
    # Differentiate between alone and not alone
    # 1 Alone 2,3,4 Small, 5,6 Medium, 7,8 Large
    df['FamilySize'] = df['FamilySize'].map(lambda x: 1 if x == 1 else (2 if x == 2 or x == 3 or x == 4 else (3 if x == 5 or x == 6 else 4)))
    
    # Mlle is Miss and Mme is Mrs, put Countess and Lady into Mrs
    df['Name'] = df['Name'].replace('Mlle', 'Miss')
    df['Name'] = df['Name'].replace(['Mme', 'Ms'], 'Mrs')
    df['Name'] = df['Name'].replace(['Countess', 'Lady'], 'Mrs')
    # Replace Capt, Don, Jonkheer as Mr
    df['Name'] = df['Name'].replace(['Capt', 'Don', 'Jonkheer'], 'Mr')
    # Dr, Col, Major, and Sir are rare and can be grouped with Rare
    df['Name'] = df['Name'].replace(['Dr', 'Col', 'Major', 'Sir'], 'Rare')


    

    #drop Fare
    df.drop('Fare', axis=1, inplace=True)




    #Change categorical to numerical
    Categorical = ['Name','Sex','Embarked','Deck']
    for cat in Categorical:
        df[cat] = pd.factorize(df[cat])[0]
        





    return df

train = clean(train)
test = clean(test)
train.head()
    

,PassengerId,Survived,Pclass,Name,Sex,Embarked,FareBin,AgeBin,Deck,FamilySize
0,1,0,3,0,0,0,1,2,0,2
1,2,1,1,1,1,1,11,7,1,2
2,3,1,3,2,1,0,3,4,0,1
3,4,1,1,1,1,0,10,7,1,2
4,5,0,3,0,0,0,3,7,0,1


In [13]:
train.Deck.unique()

array([0, 1, 2, 3], dtype=int64)

In [48]:
from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.pipeline import Pipeline

# Split data
X = train.drop(['Survived', 'PassengerId'], axis=1)
y = train['Survived']
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# KFold
kf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

# Define the pipeline
params = {
    'randomforestclassifier__n_estimators': [1750],
    'randomforestclassifier__max_depth': [7],
    'randomforestclassifier__min_samples_split': [6],
    'randomforestclassifier__min_samples_leaf': [6],
    'randomforestclassifier__bootstrap': [True, False]
}

pipe = Pipeline(steps=[('randomforestclassifier', RandomForestClassifier(random_state=42))])

# Define the grid search
search = RandomizedSearchCV(pipe, params, n_iter=100, cv=kf, scoring='accuracy', n_jobs=-1, verbose=1, random_state=42)
search.fit(X_train, y_train)

# Get the best estimator from the random search
RFC_best = search.best_estimator_

# Repeat the process for other classifiers (ExtC, SVMC, AdaBoostClassifier, GradientBoostingClassifier)
# ...

# Create the VotingClassifier
votingC = VotingClassifier(estimators=[('rfc', RFC_best), ('extc', ExtC_best),
                                        ('svc', SVMC_best), ('adac', ada_best), ('gbc', GBC_best)],
                           voting='soft', n_jobs=4)

# Fit the VotingClassifier
votingC.fit(X_train, y_train)

# Evaluate the VotingClassifier on the validation set
accuracy = votingC.score(X_val, y_val)
print(f"Validation Accuracy: {accuracy:.4f}")

# Make predictions on the test set
test_predictions = votingC.predict(test.drop(['PassengerId'], axis=1))

# Create a submission file
submission = pd.DataFrame({'PassengerId': test['PassengerId'], 'Survived': test_predictions})
submission.to_csv('submission.csv', index=False)

C:\Users\eugen\AppData\Roaming\Python\Python312\site-packages\sklearn\model_selection\_search.py:318: UserWarning: The total space of parameters 2 is smaller than n_iter=100. Running 2 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


Fitting 10 folds for each of 2 candidates, totalling 20 fits


NameError: name 'ExtC_best' is not defined

In [47]:
# Split data    
X = train.drop(['Survived','PassengerId'], axis=1)
y = train['Survived']
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

#Kfold
kf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

# Define the pipeline
params={'randomforestclassifier__n_estimators': [1750],
        'randomforestclassifier__max_depth': [7],
        'randomforestclassifier__min_samples_split': [6],
        'randomforestclassifier__min_samples_leaf': [6],
        'randomforestclassifier__bootstrap': [True, False]}

pipe = Pipeline(steps=[('randomforestclassifier', RandomForestClassifier(random_state=42))])

# Define the grid search
search = RandomizedSearchCV(pipe, params, n_iter=100, cv=kf, scoring='accuracy', n_jobs=-1, verbose=1, random_state=42)
search.fit(X_train, y_train)






votingC = VotingClassifier(estimators=[('rfc', RFC_best), ('extc', ExtC_best),
('svc', SVMC_best), ('adac',ada_best),('gbc',GBC_best)], voting='soft', n_jobs=4)

votingC = votingC.fit(X_train, Y_train)



C:\Users\eugen\AppData\Roaming\Python\Python312\site-packages\sklearn\model_selection\_search.py:318: UserWarning: The total space of parameters 2 is smaller than n_iter=100. Running 2 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


Fitting 10 folds for each of 2 candidates, totalling 20 fits


RandomizedSearchCV(cv=StratifiedKFold(n_splits=10, random_state=42, shuffle=True),
                   estimator=Pipeline(steps=[('randomforestclassifier',
                                              RandomForestClassifier(random_state=42))]),
                   n_iter=100, n_jobs=-1,
                   param_distributions={'randomforestclassifier__bootstrap': [True,
                                                                              False],
                                        'randomforestclassifier__max_depth': [7],
                                        'randomforestclassifier__min_samples_leaf': [6],
                                        'randomforestclassifier__min_samples_split': [6],
                                        'randomforestclassifier__n_estimators': [1750]},
                   random_state=42, scoring='accuracy', verbose=1)

In [40]:
predictions=search.predict(test.drop('PassengerId', axis=1))
submission = pd.DataFrame({'PassengerId': test['PassengerId'], 'Survived': predictions})
submission.to_csv('submission.csv', index=False)

In [46]:





votingC = VotingClassifier(estimators=[('rfc', RFC_best), ('extc', ExtC_best),
('svc', SVMC_best), ('adac',ada_best),('gbc',GBC_best)], voting='soft', n_jobs=4)

votingC = votingC.fit(X_train, Y_train)


C:\Users\eugen\AppData\Roaming\Python\Python312\site-packages\sklearn\model_selection\_search.py:318: UserWarning: The total space of parameters 1 is smaller than n_iter=100. Running 1 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


Fitting 10 folds for each of 1 candidates, totalling 10 fits


In [73]:
X = train.drop(["Survived","PassengerId"], axis=1)  # Dropping the target & identifier
y = train['Survived']  # Defining the target variable

# Splitting into train and test to avoid data leakage during transformation
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Identifying numerical and categorical features
numerical_features = X_train.select_dtypes(include=['int64', 'float64']).columns
categorical_features = X_train.select_dtypes(include=['object']).columns

# Creating a column transformer to apply transformations to different column types
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ])

# Creating a pipeline that first transforms the data and then fits a model
pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                           ('classifier', GradientBoostingClassifier(random_state=42))])

# Hyperparameter tuning
param_grid = {
    'classifier__n_estimators': [100, 200],
    'classifier__learning_rate': [0.01, 0.1],
    'classifier__max_depth': [3, 5]
}

search = GridSearchCV(pipeline, param_grid, n_jobs=-1, scoring='accuracy', cv=5)
search.fit(X_train, y_train)

print("Best parameter (CV score=%0.3f):" % search.best_score_)
print(search.best_params_) 

Best parameter (CV score=0.830):
{'classifier__learning_rate': 0.01, 'classifier__max_depth': 3, 'classifier__n_estimators': 200}


In [76]:

X_test_new = test.drop(['PassengerId'], axis=1, errors='ignore')
predictions = search.predict(X_test_new)        
submission = pd.DataFrame({'PassengerId': test['PassengerId'], 'Survived': predictions})
submission.to_csv('submission.csv', index=False)

In [3]:
# Preprocessing
# import data
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

# clean data
def clean(df):
    # Extract only prefix from Name and just keep prefix
    df['Name'] = df['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False)
    # Drop ticket
    df.drop('Ticket', axis=1, inplace=True)
    # Split cabin into letter and number
    df['CabinClass'] = df['Cabin'].str[0]
    df.drop('Cabin', axis=1, inplace=True)
    # Fill missing values
    imputer = IterativeImputer()
    num_columns = ['Age', 'Fare']  
    df[num_columns] = imputer.fit_transform(df[num_columns])
    # Filling missing values in CabinClass and Embarked
    df['CabinClass'] = df['CabinClass'].fillna('X')
    df['Embarked'] = df['Embarked'].fillna('X')
    # Family size
    df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
    # Drop SibSp and Parch
    df.drop(['SibSp', 'Parch'], axis=1, inplace=True)
    # Differentiate between alone and not alone
    df['IsAlone'] = 0
    df.loc[df['FamilySize'] == 1, 'IsAlone'] = 1
    # Change age into categories
    df['Age'] = pd.cut(df['Age'], bins=[0, 12, 18, 65, 100], labels=['Child', 'Teenager', 'Adult', 'Elderly'])
    # Change fare into 5 categories
    df['Fare'] = pd.qcut(df['Fare'], 5, labels=['Very Low', 'Low', 'Medium', 'High', 'Very High'])

    # Mlle is Miss and Mme is Mrs, put Countess and Lady into Mrs
    df['Name'] = df['Name'].replace('Mlle', 'Miss')
    df['Name'] = df['Name'].replace(['Mme', 'Ms'], 'Mrs')
    df['Name'] = df['Name'].replace(['Countess', 'Lady'], 'Mrs')
    # Replace Capt, Don, Jonkheer as Mr
    df['Name'] = df['Name'].replace(['Capt', 'Don', 'Jonkheer'], 'Mr')
    # Dr, Col, Major, and Sir are rare and can be grouped with Rare
    df['Name'] = df['Name'].replace(['Dr', 'Col', 'Major', 'Sir'], 'Rare')

    return df

train = clean(train)
test = clean(test)
train.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,Fare,Embarked,CabinClass,FamilySize,IsAlone
0,1,0,3,Mr,male,Adult,Very Low,S,X,2,0
1,2,1,1,Mrs,female,Adult,Very High,C,C,2,0
2,3,1,3,Miss,female,Adult,Low,S,X,1,1
3,4,1,1,Mrs,female,Adult,Very High,S,C,2,0
4,5,0,3,Mr,male,Adult,Low,S,X,1,1


In [7]:
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier, VotingClassifier
from sklearn.preprocessing import OrdinalEncoder

X = train.drop(["Survived","PassengerId"], axis=1)  # Dropping the target & identifier
y = train['Survived']  # Defining the target variable

# Splitting into train and test to avoid data leakage during transformation
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
# Convert categorical features to numeric using OrdinalEncoder
cat_features = ['Name', 'Sex', 'Age', 'Fare', 'Embarked', 'CabinClass']
ord_encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
X_train[cat_features] = ord_encoder.fit_transform(X_train[cat_features])
X_test[cat_features] = ord_encoder.transform(X_test[cat_features])

# Create interaction features
X_train['Pclass_Sex'] = X_train['Pclass'] * X_train['Sex']
X_train['Fare_Age'] = X_train['Fare'] * X_train['Age']
X_test['Pclass_Sex'] = X_test['Pclass'] * X_test['Sex']
X_test['Fare_Age'] = X_test['Fare'] * X_test['Age']

# Update the numerical features
numerical_features = X_train.select_dtypes(include=['int64', 'float64']).columns

# Update the preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features)
    ])

# Define individual classifiers
gbc = GradientBoostingClassifier(random_state=42)
rfc = RandomForestClassifier(random_state=42)

# Define the ensemble classifier
ensemble = VotingClassifier(
    estimators=[('gbc', gbc), ('rfc', rfc)],
    voting='soft'
)

# Update the pipeline with the ensemble classifier
pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                           ('classifier', ensemble)])

# Update the hyperparameter grid for tuning
param_grid = {
    'classifier__gbc__n_estimators': [100, 200],
    'classifier__gbc__learning_rate': [0.1, 0.3],
    'classifier__gbc__max_depth': [3, 5],
    'classifier__rfc__n_estimators': [100, 200],
    'classifier__rfc__max_depth': [None, 5],
    'classifier__rfc__min_samples_split': [2, 5]
}

# Perform grid search with cross-validation
search = GridSearchCV(pipeline, param_grid, n_jobs=-1, scoring='accuracy', cv=5)
search.fit(X_train, y_train)

print("Best parameter (CV score=%0.3f):" % search.best_score_)
print(search.best_params_)

Best parameter (CV score=0.838):
{'classifier__gbc__learning_rate': 0.3, 'classifier__gbc__max_depth': 3, 'classifier__gbc__n_estimators': 200, 'classifier__rfc__max_depth': 5, 'classifier__rfc__min_samples_split': 2, 'classifier__rfc__n_estimators': 200}


In [10]:

X_test_new = test.drop(['PassengerId'], axis=1, errors='ignore')
predictions = search.predict(X_test_new)        
submission = pd.DataFrame({'PassengerId': test['PassengerId'], 'Survived': predictions})
submission.to_csv('submission.csv', index=False)

In [16]:
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier, VotingClassifier
from sklearn.preprocessing import OrdinalEncoder

X = train.drop(["Survived","PassengerId"], axis=1)  # Dropping the target & identifier
y = train['Survived']  # Defining the target variable

# Splitting into train and test to avoid data leakage during transformation
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
# Convert categorical features to numeric using OrdinalEncoder
cat_features = ['Name', 'Sex', 'Age', 'Fare', 'Embarked', 'CabinClass']
ord_encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
X_train[cat_features] = ord_encoder.fit_transform(X_train[cat_features])
X_test[cat_features] = ord_encoder.transform(X_test[cat_features])

# Create interaction features
X_train['Pclass_Sex'] = X_train['Pclass'] * X_train['Sex']
X_train['Fare_Age'] = X_train['Fare'] * X_train['Age']
X_test['Pclass_Sex'] = X_test['Pclass'] * X_test['Sex']
X_test['Fare_Age'] = X_test['Fare'] * X_test['Age']

# Update the numerical features
numerical_features = X_train.select_dtypes(include=['int64', 'float64']).columns


# Create the preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features)
    ])

# Create the individual classifiers with the best parameters
gbc = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)

rfc = RandomForestClassifier(
    n_estimators=100,
    max_depth=5,
    min_samples_split=2,
    random_state=42
)

# Create the ensemble classifier
ensemble = VotingClassifier(
    estimators=[('gbc', gbc), ('rfc', rfc)],
    voting='soft'
)

# Create the final pipeline
best_pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                                ('classifier', ensemble)])

# Fit the best model on the entire training data
best_model = best_pipeline.fit(X_train, y_train)


In [17]:
# Make predictions on the test set
y_pred = best_model.predict(X_test)

# Evaluate the model's performance
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

Accuracy: 0.8156424581005587


In [18]:
# Prepare X_test_new by applying the same preprocessing steps as for the training data
X_test_new = test.drop(['PassengerId'], axis=1, errors='ignore')

# Convert categorical features to numeric using OrdinalEncoder
X_test_new[cat_features] = ord_encoder.transform(X_test_new[cat_features])

# Create interaction features
X_test_new['Pclass_Sex'] = X_test_new['Pclass'] * X_test_new['Sex']
X_test_new['Fare_Age'] = X_test_new['Fare'] * X_test_new['Age']

# Make predictions using the trained model
predictions = search.predict(X_test_new)

# Create a submission DataFrame
submission = pd.DataFrame({'PassengerId': test['PassengerId'], 'Survived': predictions})

# Save the submission DataFrame to a CSV file
submission.to_csv('submission.csv', index=False)